In [35]:
import sys
sys.path.append("/Users/livardywufianto/Projects/Trading/vscode/technical-analysis/src")

from technical_analysis.backtest import run_backtest, evaluate_run_performance

In [9]:
import pandas as pd
import os

# test_dir = "/Users/livardywufianto/Projects/Trading/data/outputs/backtest_test_v3/"
# ln_volume = False
# test_filepaths = [os.path.join(
#     test_dir, x) for x in os.listdir(test_dir) if str(ln_volume) in x]

test_filepaths = [
    "/Users/livardywufianto/Projects/Trading/data/Debugging/volume_check_2026-05-15_ln=False.csv",
    "/Users/livardywufianto/Projects/Trading/data/Debugging/volume_check_2026-05-14_ln=False.csv",
    "/Users/livardywufianto/Projects/Trading/data/Debugging/volume_check_2026-05-13_ln=False.csv",
    "/Users/livardywufianto/Projects/Trading/data/Debugging/volume_check_2026-05-12_ln=False.csv", 
    "/Users/livardywufianto/Projects/Trading/data/Debugging/volume_check_2026-05-11_ln=False.csv",
]

test_filepaths = [
    "/Users/livardywufianto/Projects/Trading/data/outputs/backtest_test_v3/volume_check_2026-05-15_ln=False.csv",
    "/Users/livardywufianto/Projects/Trading/data/outputs/backtest_test_v3/volume_check_2026-05-14_ln=False.csv",
    "/Users/livardywufianto/Projects/Trading/data/outputs/backtest_test_v3/volume_check_2026-05-13_ln=False.csv",
    "/Users/livardywufianto/Projects/Trading/data/outputs/backtest_test_v3/volume_check_2026-05-12_ln=False.csv", 
    "/Users/livardywufianto/Projects/Trading/data/outputs/backtest_test_v3/volume_check_2026-05-11_ln=False.csv",
]



df_list = [pd.read_csv(fp) for fp in test_filepaths]
test_df = pd.concat(df_list, ignore_index=True)

In [10]:
len(test_df), len(test_df.drop_duplicates())

(65676, 65676)

# Buy Signal

In [30]:
import pandas as pd
import numpy as np

def look_backward_per_ticker(df: pd.DataFrame, column_to_shift: str, ticker_col: str = 'ticker', periods: int = 1) -> pd.Series:
    """
    Shifts a column backward by a set number of periods, isolated by ticker.
    Guarantees no data leakage between different stocks.
    """
    # Group by ticker, select the column, and apply the shift within that group
    return df.groupby(ticker_col)[column_to_shift].shift(periods)

def apply_mean_reversion_rule(df, z_score_threshold=2, body_ratio_threshold=0.7):
    condition_panic_selling = (df['IS_RED'] == True) & (df['BODY_RATIO'] >= body_ratio_threshold)
    condition_stat_extreme  = (df['close'] <= df['bb_lower']) & (df['z_score'].abs() > z_score_threshold)
    above_sma = buy_df["close"] >= buy_df["sma_20"]
    
    df['panic_alert'] = condition_panic_selling & condition_stat_extreme & above_sma
    # df['was_previous_hour_panic'] = look_backward_per_ticker(df, column_to_shift='panic_alert', periods=1)
    # df['buy_signal'] = (df['was_previous_hour_panic'] == True) & (df['IS_RED'] == False)
    df["buy_signal"] = df["panic_alert"]
    return df

z_score_threshold = 0.2
body_ratio_threshold = 0.7
profit_target = 0.03
stop_loss_target = 0.03
max_held_hours = 48

params_dict = {
    "z_score_threshold": z_score_threshold,
    "body_ratio_threshold": body_ratio_threshold,
    "profit_target": profit_target,
    "stop_loss_target": stop_loss_target,
    "max_held_hours": max_held_hours,
}

buy_df = apply_mean_reversion_rule(
    test_df, z_score_threshold=z_score_threshold,
    body_ratio_threshold=body_ratio_threshold
)

In [34]:
buy_df[buy_df["buy_signal"]]

,ticker,us_date,hour,open,high,low,close,volume,transactions,volume_mean,...,IS_RED,BODY_RATIO,IS_BODY_LARGE,bb_upper,bb_middle,bb_lower,ema_20,sma_20,panic_alert,buy_signal


# Backtest

In [5]:
"""
Possible Exit Reason:
- Take_Profit
- Stop_Loss
- Time_Limit_48h
- Data_Cutoff_End_of_file
- Stop_Loss_Simultaneous_Hit (TP and SL hit at the same time)
"""
backtest_df = run_backtest(
    buy_df, 
    params_dict
)

# Evaluation

In [6]:
evaluate_run_performance(backtest_df, params_dict)

{'z_score_threshold': 0.2,
 'body_ratio_threshold': 0.7,
 'profit_target': 0.03,
 'stop_loss_target': 0.03,
 'max_held_hours': 48,
 'total_trades': 1440,
 'win_rate': 28.26,
 'stop_rate': 22.99,
 'timeout_rate': 23.82,
 'cutoff_ratet': 24.65,
 'win_rate_adjusted': 37.65,
 'stop_rate_adjusted': 37.65,
 'timeout_rate_adjusted': 31.73,
 'profit_factor': 1.48,
 'avg_holding_hours': 35.6}

In [26]:
buy_df.columns

Index(['ticker', 'us_date', 'hour', 'open', 'high', 'low', 'close', 'volume',
       'transactions', 'volume_mean', 'volume_std', 'z_score', 'IS_RED',
       'BODY_RATIO', 'IS_BODY_LARGE', 'bb_upper', 'bb_middle', 'bb_lower',
       'ema_20', 'sma_20', 'panic_alert', 'buy_signal'],
      dtype='object')